# Sentiment Analysis

Aitezaz Alam

In this project, I classify movie reviews as positive or negative. I use TF-IDF with logistic regression and a small PyTorch neural network.

Dataset: [IMDb movie reviews](https://huggingface.co/datasets/stanfordnlp/imdb).

## Install libraries

These are the libraries used in this project.

In [ ]:
%pip install -q datasets scikit-learn pandas matplotlib joblib torch

## Import libraries

I set a random seed and choose GPU if it is available.

In [ ]:
import copy, json, random, re, html
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_FEATURES = 5000
HIDDEN_DIM = 64
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 0.001
print('Device:', device)

## Load the dataset

I use IMDb movie reviews. The text is the input and the label is the target. Label 0 means negative and label 1 means positive.

In [ ]:
raw_train, raw_test = load_dataset('stanfordnlp/imdb', split=['train', 'test'])
df_train = raw_train.to_pandas()
df_test = raw_test.to_pandas()
print('Original shapes:', df_train.shape, df_test.shape)
display(df_train.head())
print('Missing values:\n', df_train.isna().sum())
print('Label counts:\n', df_train['label'].value_counts())
print('Duplicate raw review texts:', df_train['text'].duplicated().sum())
print('Example review:\n', df_train['text'].iloc[0])

## Clean the reviews

I remove HTML tags, empty reviews and repeated reviews. I also remove reviews with conflicting labels and test reviews that appear in the training data.

In [ ]:
def clean_text(text):
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def prepare_reviews(frame):
    frame = frame.dropna(subset=['text', 'label']).copy()
    frame['text'] = frame['text'].map(clean_text)
    frame = frame[frame['text'].str.len() > 0].copy()
    frame['key'] = frame['text'].str.lower()
    return frame

train_clean = prepare_reviews(df_train)
test_clean = prepare_reviews(df_test)
label_counts = pd.concat([train_clean, test_clean]).groupby('key')['label'].nunique()
conflicting_keys = set(label_counts[label_counts > 1].index)
print('Conflicting normalized texts excluded:', len(conflicting_keys))
train_clean = train_clean[~train_clean['key'].isin(conflicting_keys)].drop_duplicates('key')
test_clean = test_clean[~test_clean['key'].isin(conflicting_keys)].drop_duplicates('key')
overlap = test_clean['key'].isin(set(train_clean['key']))
print('Train/test overlapping texts removed from test:', int(overlap.sum()))
test_clean = test_clean[~overlap].reset_index(drop=True)
train_clean = train_clean.reset_index(drop=True)
print('Cleaned shapes:', train_clean.shape, test_clean.shape)
assert set(train_clean['label'].unique()) == {0, 1}
assert set(test_clean['label'].unique()) == {0, 1}

## Split the data

I keep 20% of the training reviews for validation. I use this set to choose the model and keep the test set for the final comparison.

In [ ]:
train_df, val_df = train_test_split(
    train_clean, test_size=0.2, random_state=SEED,
    stratify=train_clean['label'])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
y_train = train_df['label'].to_numpy(dtype=np.int64)
y_val = val_df['label'].to_numpy(dtype=np.int64)
y_test = test_clean['label'].to_numpy(dtype=np.int64)
print('Train / validation / test:', len(train_df), len(val_df), len(test_clean))
print('Validation label proportions:\n', val_df['label'].value_counts(normalize=True))
train_df['label'].value_counts().sort_index().plot.bar()
plt.xticks([0, 1], ['Negative', 'Positive'], rotation=0)
plt.ylabel('Training reviews')
plt.title('Training class balance')
plt.show()

## TF-IDF

TF-IDF converts the reviews into numbers. I use words and two-word phrases, with up to 5,000 features. I fit it only on the training reviews.

In [ ]:
tfidf = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1, 2),
                        min_df=2, dtype=np.float32)
X_train = tfidf.fit_transform(train_df['text'])
X_val = tfidf.transform(val_df['text'])
X_test = tfidf.transform(test_clean['text'])
print('Feature shapes:', X_train.shape, X_val.shape, X_test.shape)
print('Vocabulary sample:', tfidf.get_feature_names_out()[:20])
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]

## Logistic regression

I train logistic regression first. This gives me a simple model to compare with the neural network.

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
baseline_val_pred = baseline.predict(X_val)
baseline_val_f1 = f1_score(y_val, baseline_val_pred, zero_division=0)
print('Baseline validation accuracy:', accuracy_score(y_val, baseline_val_pred))
print('Baseline validation F1:', baseline_val_f1)

## Prepare batches

I divide the data into batches. Only one batch is converted into dense tensors at a time to reduce memory use.

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return index, float(self.labels[index])

def make_loader(features, labels, shuffle=False):
    data = ReviewDataset(features, labels)
    def collate_batch(batch):
        indices, targets = zip(*batch)
        x = torch.from_numpy(features[list(indices)].toarray().astype(np.float32, copy=False))
        y = torch.tensor(targets, dtype=torch.float32).reshape(-1, 1)
        return x, y
    return DataLoader(data, batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=collate_batch, num_workers=0)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test)
xb, yb = next(iter(train_loader))
print('Batch shapes:', xb.shape, yb.shape)

## Create the neural network

The network has 64 hidden neurons, ReLU and dropout. It returns one score for each review. I use BCEWithLogitsLoss and Adam for training.

In [ ]:
class SentimentModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1))
    def forward(self, x):
        return self.network(x)

model = SentimentModel(X_train.shape[1], HIDDEN_DIM).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(model)
print('Trainable parameters:', sum(p.numel() for p in model.parameters()))

## Train the neural network

I train the network and check validation F1 after each epoch. I keep the best weights and stop if F1 does not improve for three epochs.

In [ ]:
def evaluate_network(network, loader):
    network.eval()
    total_loss = 0.0
    probabilities, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = network(xb)
            total_loss += loss_fn(logits, yb).item() * len(yb)
            probabilities.append(torch.sigmoid(logits).cpu().numpy().ravel())
            targets.append(yb.cpu().numpy().ravel())
    probabilities = np.concatenate(probabilities)
    targets = np.concatenate(targets).astype(int)
    return total_loss / len(targets), probabilities, targets

history = []
best_f1 = -1.0
best_state = None
best_epoch = 0
patience, stale_epochs = 3, 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item() * len(yb)
    train_loss = total_train_loss / len(y_train)
    val_loss, val_probs, val_targets = evaluate_network(model, val_loader)
    val_pred = (val_probs >= 0.5).astype(int)
    val_f1 = f1_score(val_targets, val_pred, zero_division=0)
    val_acc = accuracy_score(val_targets, val_pred)
    history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss,
                        val_accuracy=val_acc, val_f1=val_f1))
    print(f'Epoch {epoch:02d}: train loss={train_loss:.4f}, '
          f'val loss={val_loss:.4f}, val accuracy={val_acc:.4f}, val F1={val_f1:.4f}')
    if val_f1 > best_f1:
        best_f1, best_epoch = val_f1, epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale_epochs = 0
    else:
        stale_epochs += 1
        if stale_epochs >= patience:
            print('Early stopping: validation F1 has not improved.')
            break
model.load_state_dict(best_state)
print('Restored epoch:', best_epoch)
selected_model = 'pytorch' if best_f1 > baseline_val_f1 else 'logistic_regression'
print('Selected using validation F1:', selected_model)

## Training graphs

These graphs show training loss, validation loss and validation F1. They help me see how the model learns.

In [ ]:
history_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='Training')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='Validation')
axes[0].set(xlabel='Epoch', ylabel='BCE loss', title='Loss curves')
axes[0].legend()
axes[1].plot(history_df['epoch'], history_df['val_f1'])
axes[1].set(xlabel='Epoch', ylabel='F1', title='Validation F1')
plt.tight_layout()
plt.show()

## Evaluate the models

I compare accuracy and F1 on the cleaned test set. I also show confusion matrices and some wrong predictions. I had already checked test results in an earlier attempt.

In [ ]:
baseline_test_pred = baseline.predict(X_test)
_, nn_test_probs, test_targets = evaluate_network(model, test_loader)
nn_test_pred = (nn_test_probs >= 0.5).astype(int)
results = pd.DataFrame([
    {'model': 'logistic_regression', 'validation_f1': baseline_val_f1,
     'test_accuracy': accuracy_score(y_test, baseline_test_pred),
     'test_f1': f1_score(y_test, baseline_test_pred, zero_division=0)},
    {'model': 'pytorch', 'validation_f1': best_f1,
     'test_accuracy': accuracy_score(y_test, nn_test_pred),
     'test_f1': f1_score(y_test, nn_test_pred, zero_division=0)}])
display(results)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name, pred in zip(axes, ['Logistic regression', 'PyTorch'],
                          [baseline_test_pred, nn_test_pred]):
    print(name)
    print(classification_report(y_test, pred, labels=[0, 1],
                                target_names=['Negative', 'Positive'], zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=[0, 1],
        display_labels=['Negative', 'Positive'], ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()
final_test_pred = nn_test_pred if selected_model == 'pytorch' else baseline_test_pred
errors = test_clean.loc[final_test_pred != y_test, ['text', 'label']].copy()
errors['predicted_label'] = final_test_pred[final_test_pred != y_test]
display(errors.head(5))
if len(errors):
    print('Read one complete mistake:\n', errors['text'].iloc[0])

## Try new reviews

I test the selected model on new reviews. It predicts positive or negative. The probability is a model estimate, and there is no neutral class.

In [ ]:
def predict_reviews(reviews, network=model, vectorizer=tfidf,
                    logistic=baseline, choice=selected_model):
    cleaned = [clean_text(text) for text in reviews]
    if any(not text for text in cleaned):
        raise ValueError('Please provide nonempty reviews.')
    features = vectorizer.transform(cleaned)
    if choice == 'pytorch':
        network.eval()
        network_device = next(network.parameters()).device
        chunks = []
        with torch.no_grad():
            for start in range(0, len(reviews), BATCH_SIZE):
                xb = torch.from_numpy(features[start:start+BATCH_SIZE].toarray().astype(np.float32))
                chunks.append(torch.sigmoid(network(xb.to(network_device))).cpu().numpy().ravel())
        probabilities = np.concatenate(chunks)
    else:
        positive_column = list(logistic.classes_).index(1)
        probabilities = logistic.predict_proba(features)[:, positive_column]
    return pd.DataFrame({'review': reviews, 'positive_probability': probabilities,
        'prediction': np.where(probabilities >= 0.5, 'Positive', 'Negative')})

examples = ['The acting was excellent and I enjoyed the story.',
            'A boring movie with a terrible ending.',
            'The movie was not good.',
            'The acting was good, but the story was disappointing.']
display(predict_reviews(examples))

## Save the models

I save the vectorizer, both models and the results. I also create a README. In Colab, these files need to be downloaded or saved to Drive before the session ends.

In [ ]:
OUTPUT_DIR = Path('/content/sentiment_artifacts') if Path('/content').exists() else Path('sentiment_artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(tfidf, OUTPUT_DIR / 'tfidf_vectorizer.joblib')
joblib.dump(baseline, OUTPUT_DIR / 'logistic_regression.joblib')
torch.save(best_state, OUTPUT_DIR / 'sentiment_network_weights.pt')
config = {'input_dim': int(X_train.shape[1]), 'hidden_dim': HIDDEN_DIM,
          'dropout': 0.3, 'threshold': 0.5, 'selected_model': selected_model,
          'best_epoch': best_epoch, 'seed': SEED,
          'split_sizes': {'train': len(train_df), 'validation': len(val_df), 'test': len(test_clean)},
          'test_note': 'Cleaned subset; test performance was viewed in an earlier attempt.'}
(OUTPUT_DIR / 'config.json').write_text(json.dumps(config, indent=2))
results.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
history_df.to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
readme = f"""# Sentiment Analysis

Aitezaz Alam

This project predicts whether an English movie review is positive or negative.
I use IMDb reviews, TF-IDF, logistic regression and a small PyTorch neural network.

## Dataset
https://huggingface.co/datasets/stanfordnlp/imdb

I clean HTML and duplicate reviews, then split the training data into training and validation sets.
The test results use a cleaned subset. I had checked the original test results in an earlier attempt.

## Results
{results.to_string(index=False)}

Selected model based on validation F1: {selected_model}

## How to run
Open the notebook in Google Colab and run the cells in order.
The notebook includes prediction and saving/loading both models.

## Limitations
The model has only positive and negative classes. It may make mistakes on sarcasm,
mixed opinions or text that is different from movie reviews.
"""
(OUTPUT_DIR / 'README.md').write_text(readme)
import importlib.metadata
packages = ['datasets', 'numpy', 'pandas', 'scikit-learn', 'matplotlib', 'joblib', 'torch']
(OUTPUT_DIR / 'requirements.txt').write_text('\n'.join(
    f'{p}=={importlib.metadata.version(p)}' for p in packages))
print('Saved to:', OUTPUT_DIR.resolve())

## Load the saved models

I load the saved files and check that the predictions match. This allows me to use the model without training again.

In [ ]:
loaded_vectorizer = joblib.load(OUTPUT_DIR / 'tfidf_vectorizer.joblib')
loaded_baseline = joblib.load(OUTPUT_DIR / 'logistic_regression.joblib')
loaded_config = json.loads((OUTPUT_DIR / 'config.json').read_text())
loaded_network = SentimentModel(loaded_config['input_dim'], loaded_config['hidden_dim'],
                                loaded_config['dropout'])
loaded_network.load_state_dict(torch.load(OUTPUT_DIR / 'sentiment_network_weights.pt',
                                          map_location='cpu', weights_only=True))
loaded_network.eval()
before = predict_reviews(examples)
after = predict_reviews(examples, network=loaded_network,
                        vectorizer=loaded_vectorizer, logistic=loaded_baseline,
                        choice=loaded_config['selected_model'])
np.testing.assert_allclose(before['positive_probability'], after['positive_probability'],
                           rtol=1e-4, atol=1e-5)
assert before['prediction'].tolist() == after['prediction'].tolist()
print('Reload verification passed.')
display(after)
import shutil
zip_path = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
print('ZIP:', zip_path)
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(zip_path)

## Conclusion

I compare both models and select the final model using validation F1. The results are shown above. The model can make mistakes on sarcasm and reviews with mixed opinions. It is trained on English movie reviews, so it may not work as well on other types of text.